# Praetor — SFT + GRPO training notebook

This notebook trains the **Praetor** incident-response agent end-to-end:

1. **SFT** (supervised fine-tune) the policy on 16 senior-SRE behavioral-clone
   trajectories drawn from `IDEAL_TRAJECTORIES` in `coach.py`.
2. **GRPO** (Group Relative Policy Optimization) the resulting policy against
   our 6-component verifiable reward, with curriculum-driven scenario sampling.
3. **Evaluate** all 8 scenario families across random / SFT / SFT+GRPO.
4. **Render** the 4 canonical plots into `/content/results/` for the README.
5. **Push** the LoRA adapter to your HuggingFace Hub.

### Total runtime budget

| Phase | A100 | T4 |
|---|---:|---:|
| Setup + clone | 2–3 min | 2–3 min |
| Load model | 2–3 min | 3–5 min |
| SFT (1 epoch) | 30–40 min | 75–90 min |
| SFT sanity-eval (9 episodes) | ~2 min | ~5 min |
| GRPO (A100 200 steps / T4 30 steps) | 2–3 hr | ~60 min |
| Final eval (120 episodes) | 5–7 min | 15–20 min |
| Plots + push | 3–5 min | 3–5 min |
| **Total** | **~3.5 hr** | **~2.5–3 hr** |

T4's GRPO budget is intentionally trimmed (30 steps vs A100's 200) to fit a
~3-hour total wall-clock. 30 steps is enough to see the policy improving
across the 6 reward components — the trend is visible, the killer per-component
plot still tells the story — even if the policy hasn't fully converged. If
you have more time, raise `GRPO_STEPS` in cell 2.

### Before you run

- **Runtime → Change runtime type → A100 GPU** (or T4 if A100 unavailable).
- **Run all** from the menu, then walk away. The notebook writes intermediate
  state to `/content/checkpoints/` so a disconnect during GRPO doesn't lose
  the SFT work.
- If any cell errors, the next cell will refuse to start. Fix the error,
  re-run the failing cell, then continue.


## Cell 1 — Setup: install dependencies + clone the repo

**Expected runtime: 2–3 minutes (any GPU).** Most of the wall-clock is the
`unsloth` install and a single git clone.


In [ ]:
import time, os, sys, subprocess
_t0 = time.monotonic()

# Install pinned versions known to work together with Unsloth + TRL + the
# bitsandbytes 4-bit kernels on Colab's image. Kept silent so the cell output
# is dominated by useful info, not pip noise.
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" 2>&1 | tail -1
%pip install -q --no-deps "trl<0.16" "peft>=0.10" "accelerate>=0.30" "bitsandbytes>=0.43" 2>&1 | tail -1
# transformers pinned <4.50 because newer versions changed the
# `_get_train_sampler` signature in a way Unsloth's GRPOTrainer override
# doesn't yet handle (TypeError: takes 1 positional argument but 2 given).
%pip install -q "transformers>=4.45,<4.50" "datasets>=2.14" "huggingface-hub>=0.20" "matplotlib>=3.7" 2>&1 | tail -1
%pip install -q "pydantic>=2.0" "fastapi>=0.104" 2>&1 | tail -1

# Clone (or update) the project so imports like `training.eval_runner` resolve.
if not os.path.exists("/content/incident-commander"):
    !git clone -q https://github.com/root4shreshth/incident-commander.git /content/incident-commander
else:
    !cd /content/incident-commander && git pull -q
sys.path.insert(0, "/content/incident-commander")

os.makedirs("/content/checkpoints", exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

print(f"Setup OK in {time.monotonic()-_t0:.1f}s. CWD: /content/incident-commander")


## Cell 2 — Compute detection

**Expected runtime: < 1 second.** Reports the GPU and adjusts later cells'
batch sizes accordingly.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Runtime → Change runtime type → A100 (or T4)."
    )
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
IS_A100 = "A100" in GPU_NAME

# Batch size + GRPO step counts scale with GPU memory.
SFT_BATCH = 4 if IS_A100 else 2
SFT_GRAD_ACCUM = 4 if IS_A100 else 8        # effective batch = 16 either way
GRPO_STEPS = 200 if IS_A100 else 30   # T4 cut from 120 to fit ~3 hr total
GRPO_NUM_GENERATIONS = 4 if IS_A100 else 2  # rollouts per prompt
GRPO_BATCH = 2 if IS_A100 else 1

print(f"GPU            : {GPU_NAME} ({GPU_MEM_GB:.1f} GB)")
print(f"SFT batch      : {SFT_BATCH} × grad_accum {SFT_GRAD_ACCUM} (eff. {SFT_BATCH*SFT_GRAD_ACCUM})")
print(f"GRPO steps     : {GRPO_STEPS}")
print(f"GRPO rollouts  : {GRPO_NUM_GENERATIONS} per prompt")


## Cell 3 — Load Qwen2.5-Coder-1.5B in 4-bit with LoRA

**Expected runtime: 2–3 minutes on A100, 3–5 minutes on T4.** Downloads
~1.5 GB of model weights, applies the 4-bit quantization, and attaches
a LoRA r=16 adapter ready for SFT.

Skip-if-already-done: if `model` is already in scope (re-running the
notebook after a partial run), this cell is a no-op.


In [ ]:
import time
_t0 = time.monotonic()

if "model" not in globals():
    from unsloth import FastLanguageModel

    MODEL_NAME = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit"
    MAX_SEQ_LEN = 4096

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        dtype=None,  # auto: bf16 on A100, fp16 on T4
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16, lora_alpha=32, lora_dropout=0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    print(f"Model loaded + LoRA attached in {time.monotonic()-_t0:.1f}s")
else:
    print("Model already loaded; skipping.")


## Cell 4 — Build the SFT chat dataset

**Expected runtime: < 30 seconds.** Materializes ~120 (system, user, assistant)
chat rows from `IDEAL_TRAJECTORIES` in `incident_commander_env/server/coach.py`,
sampled across multiple seeds per family so each scenario gets several
parametric variants.


In [ ]:
import time
_t0 = time.monotonic()

from training.datasets import build_sft_dataset, to_chat_messages, SYSTEM_PROMPT
from datasets import Dataset

raw_rows = build_sft_dataset(n_seeds_per_family=8)
hf_rows = []
for r in raw_rows:
    msgs = to_chat_messages(r)
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    hf_rows.append({"text": text})

sft_ds = Dataset.from_list(hf_rows)
print(f"SFT dataset: {len(sft_ds)} rows  ({time.monotonic()-_t0:.1f}s)")
print(f"  example length (chars): {len(sft_ds[0]['text'])}")
print(f"  scenarios covered     : {sorted({r['scenario'] for r in raw_rows})}")


## Cell 5 — SFT training (1 epoch)

**Expected runtime: 30–40 minutes on A100, 75–90 minutes on T4.** This is
the longest training cell; you'll see TRL's per-step progress bar.

Skip-if-already-done: if `/content/checkpoints/sft.done` exists from a
previous successful run, the cell loads weights from
`/content/checkpoints/sft_adapter/` instead of re-training.


In [ ]:
import os, time
_t0 = time.monotonic()

SFT_CHECKPOINT = "/content/checkpoints/sft_adapter"
SFT_DONE = "/content/checkpoints/sft.done"

if os.path.exists(SFT_DONE) and os.path.exists(SFT_CHECKPOINT):
    print("SFT already complete (sft.done exists). Loading adapter from checkpoint.")
    model.load_adapter(SFT_CHECKPOINT, adapter_name="default", is_trainable=True)
else:
    from trl import SFTTrainer, SFTConfig

    sft_args = SFTConfig(
        output_dir="/content/sft_runs",
        per_device_train_batch_size=SFT_BATCH,
        gradient_accumulation_steps=SFT_GRAD_ACCUM,
        num_train_epochs=1,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        logging_steps=5,
        save_strategy="no",
        bf16=IS_A100, fp16=not IS_A100,
        max_seq_length=2048,
        dataset_text_field="text",
        report_to="none",
        seed=42,
    )
    sft_trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=sft_ds, args=sft_args,
    )
    sft_trainer.train()
    model.save_pretrained(SFT_CHECKPOINT)
    tokenizer.save_pretrained(SFT_CHECKPOINT)
    open(SFT_DONE, "w").write("done")

print(f"SFT phase complete in {time.monotonic()-_t0:.1f}s")


## Cell 6 — SFT sanity-eval (9 episodes)

**Expected runtime: ~2 minutes on A100, ~5 minutes on T4.** Quick
random-vs-SFT comparison on 3 seeds × 3 families to confirm SFT learned
something before we spend hours on GRPO.

If SFT scores are below random, abort and inspect the SFT loss curve.


In [ ]:
import time, warnings, logging
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")
warnings.filterwarnings("ignore", category=FutureWarning,
                        module=r"transformers\.modeling_attn_mask_utils")
logging.getLogger("transformers").setLevel(logging.ERROR)

from training.eval_runner import evaluate, random_policy, hf_pipeline_policy
from training.datasets import SYSTEM_PROMPT
from transformers import GenerationConfig
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

# Clear Qwen's default max_length=32768 + use the right stop tokens.
# This is the fix that turned 90-minute hangs into 2-minute evals.
if hasattr(model, "generation_config") and model.generation_config is not None:
    model.generation_config.max_length = None
_eos_ids = [tokenizer.eos_token_id]
_im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
if _im_end is not None and _im_end != tokenizer.unk_token_id and _im_end not in _eos_ids:
    _eos_ids.append(_im_end)

GEN_CFG = GenerationConfig(
    max_new_tokens=160,
    do_sample=False,
    eos_token_id=_eos_ids,
    pad_token_id=tokenizer.eos_token_id,
)

def hf_generate(messages, max_new=160):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]
    out = model.generate(**inputs, generation_config=GEN_CFG)
    # Decode ONLY the newly-generated tokens. Slicing on token IDs avoids
    # the bug where `full[len(text):]` fails because tokenize=False keeps
    # special tokens in `text` but skip_special_tokens=True strips them
    # from `full` — so `full.startswith(text)` is False and the whole
    # transcript (system prompt + user + assistant) gets returned, which
    # then makes the JSON parser pick up the EXAMPLE inside the system
    # prompt instead of the model's actual response.
    new_tokens = out[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

SANITY_FAMILIES = ["oom_crash", "db_pool_exhaustion", "bad_deployment_cascade"]
SANITY_SEEDS = [2000, 2001, 2002]
TOTAL = len(SANITY_FAMILIES) * len(SANITY_SEEDS)

def _progress(idx, total, family, seed, ep):
    tag = "OK" if ep.resolved else "X "
    print(f"  [{idx:>2}/{total}] [{tag}] {family:<25s} seed={seed} steps={ep.steps_used:>2} score={ep.score:.2f}", flush=True)

_t0 = time.monotonic()
print("→ random baseline")
report_random = evaluate("random", random_policy(rng_seed=99),
                         SANITY_FAMILIES, SANITY_SEEDS,
                         system_prompt=SYSTEM_PROMPT, on_episode=_progress)

print(f"\n→ SFT model")
report_sft = evaluate("sft", hf_pipeline_policy(hf_generate),
                      SANITY_FAMILIES, SANITY_SEEDS,
                      system_prompt=SYSTEM_PROMPT, on_episode=_progress)

print(f"\nSanity eval done in {time.monotonic()-_t0:.1f}s")
print("=" * 60)
for cond, rpt in [("random", report_random), ("sft", report_sft)]:
    print(f"\n  {cond}:")
    for fam, stats in rpt.by_family.items():
        print(f"    {fam:<25s}: success={stats['success_rate']*100:>4.0f}%  "
              f"score={stats['avg_score']:.2f}  steps={stats['avg_steps_used']:.1f}")


## Cell 7 — Build the GRPO prompt dataset

**Expected runtime: < 30 seconds.** GRPO needs a list of prompts to roll
out completions against. We build one prompt per `(scenario_family, seed)`
combination drawn from a **curriculum** — easier scenarios appear earlier,
harder ones appear later.


In [ ]:
import time, random
_t0 = time.monotonic()

from training.curriculum import Curriculum
from training.datasets import SYSTEM_PROMPT
from datasets import Dataset
from incident_commander_env.server.environment import IncidentCommanderEnv

curriculum = Curriculum(rng_seed=42)

# Build one prompt per training step. The reward function reads task_id /
# seed / difficulty back out of kwargs to evaluate the completion.
prompts = []
seed_offset = 5000
for step in range(GRPO_STEPS):
    family, difficulty = curriculum.draw(step)
    seed = seed_offset + step
    env = IncidentCommanderEnv()
    obs = env.reset(task_id=family, seed=seed, difficulty=difficulty)
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"INCIDENT ALERT:\n{obs.message}\n\nBegin investigation. What is your first action?"},
    ]
    prompts.append({
        "prompt": prompt,
        "task_id": family,
        "seed": seed,
        "difficulty": difficulty,
    })

grpo_ds = Dataset.from_list(prompts)
print(f"GRPO dataset: {len(grpo_ds)} prompts  ({time.monotonic()-_t0:.1f}s)")
print(f"  family distribution: {dict((f, sum(1 for p in prompts if p['task_id'] == f)) for f in {p['task_id'] for p in prompts})}")


## Cell 8 — GRPO training

**Expected runtime: 2–3 hours on A100 (200 steps), ~60 minutes on T4 (30 steps).** This is the
longest cell. You'll see per-step reward + KL-divergence in the progress
output. The 6-component reward breakdown is logged to wandb if `WANDB_API_KEY`
is set.

Skip-if-already-done: if `/content/checkpoints/grpo.done` exists, loads
the saved adapter instead of re-training.


In [ ]:
import os, time
_t0 = time.monotonic()

GRPO_CHECKPOINT = "/content/checkpoints/grpo_adapter"
GRPO_DONE = "/content/checkpoints/grpo.done"

if os.path.exists(GRPO_DONE) and os.path.exists(GRPO_CHECKPOINT):
    print("GRPO already complete (grpo.done exists). Loading adapter from checkpoint.")
    # Replace the adapter with the GRPO version
    model.load_adapter(GRPO_CHECKPOINT, adapter_name="grpo", is_trainable=True)
    model.set_adapter("grpo")
else:
    from trl import GRPOConfig, GRPOTrainer
    from training.grpo_reward import grpo_reward_fn, reset_history

    # Defensive monkeypatch: if the runtime is on transformers >=4.50 (which
    # calls `sampler_fn(dataset)` with one positional arg) and the running
    # Unsloth still expects `_get_train_sampler(self)` only, the dataloader
    # build crashes with a TypeError. We wrap the bound method to swallow
    # any extra positional/keyword args. No-op if the signatures already match.
    try:
        from unsloth_compiled_cache.UnslothGRPOTrainer import _UnslothGRPOTrainer
        _orig_sampler = _UnslothGRPOTrainer._get_train_sampler
        def _patched_sampler(self, *args, **kwargs):
            return _orig_sampler(self)
        _UnslothGRPOTrainer._get_train_sampler = _patched_sampler
        print("Applied Unsloth GRPO sampler-signature monkeypatch.")
    except (ImportError, AttributeError):
        pass

    reset_history()  # clear sidecar between runs

    grpo_args = GRPOConfig(
        output_dir="/content/grpo_runs",
        per_device_train_batch_size=GRPO_BATCH,
        gradient_accumulation_steps=4,
        num_generations=GRPO_NUM_GENERATIONS,
        max_completion_length=160,
        max_prompt_length=2048,
        max_steps=GRPO_STEPS,
        learning_rate=5e-6,
        beta=0.04,                # KL penalty
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_steps=5,
        save_strategy="no",
        bf16=IS_A100, fp16=not IS_A100,
        report_to="none",
        seed=42,
    )

    grpo_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=grpo_reward_fn,
        args=grpo_args,
        train_dataset=grpo_ds,
    )
    grpo_trainer.train()
    model.save_pretrained(GRPO_CHECKPOINT)
    tokenizer.save_pretrained(GRPO_CHECKPOINT)
    open(GRPO_DONE, "w").write("done")

elapsed = time.monotonic() - _t0
print(f"\nGRPO phase complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")


## Cell 9 — Final eval (random vs SFT+GRPO)

**Expected runtime: 5–7 minutes on A100, 12–15 minutes on T4.**
6 families × 10 held-out seeds × 2 conditions = 120 episodes total.


In [ ]:
import time
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
if hasattr(model, "generation_config") and model.generation_config is not None:
    model.generation_config.max_length = None

EVAL_FAMILIES = [
    "oom_crash", "db_pool_exhaustion", "bad_deployment_cascade",
    "disk_full", "slow_query", "cert_expiry",
]
FINAL_SEEDS = list(range(8000, 8010))   # 10 held-out seeds
TOTAL_FINAL = len(EVAL_FAMILIES) * len(FINAL_SEEDS)

print(f"Final eval — {TOTAL_FINAL} episodes per condition × 2 conditions = {TOTAL_FINAL*2} total\n")

_t0 = time.monotonic()
print("→ random baseline")
report_random_final = evaluate("random", random_policy(rng_seed=12345),
                               EVAL_FAMILIES, FINAL_SEEDS,
                               system_prompt=SYSTEM_PROMPT, on_episode=_progress)
print(f"  random done in {time.monotonic()-_t0:.1f}s\n")

_t1 = time.monotonic()
print("→ SFT+GRPO model")
report_grpo = evaluate("sft+grpo", hf_pipeline_policy(hf_generate),
                       EVAL_FAMILIES, FINAL_SEEDS,
                       system_prompt=SYSTEM_PROMPT, on_episode=_progress)
print(f"  sft+grpo done in {time.monotonic()-_t1:.1f}s")

print("\n" + "=" * 70)
print("Final results (success rate per family):")
print("=" * 70)
print(f"{'family':<28s}  {'random':>8s}  {'sft+grpo':>10s}  {'delta':>8s}")
for fam in EVAL_FAMILIES:
    r_rate = report_random_final.by_family.get(fam, {}).get("success_rate", 0) * 100
    g_rate = report_grpo.by_family.get(fam, {}).get("success_rate", 0) * 100
    delta = g_rate - r_rate
    sign = "+" if delta >= 0 else ""
    print(f"{fam:<28s}  {r_rate:>6.0f}%  {g_rate:>8.0f}%  {sign}{delta:>5.0f}pp")


## Cell 10 — Render the 4 canonical plots

**Expected runtime: < 1 minute.** Saves all four to `/content/results/` so
you can download them directly into the README.


In [ ]:
import time, json
from collections import defaultdict
from training.plots import (
    make_reward_curve, make_reward_components,
    make_success_bars, make_action_distribution, save_figure,
)
from training.grpo_reward import get_recent_breakdowns

_t0 = time.monotonic()
out_dir = "/content/results"

# 1) GRPO training reward curve (from grpo_trainer.state.log_history if present)
try:
    log = grpo_trainer.state.log_history if "grpo_trainer" in globals() else []
    rewards = [entry.get("reward") for entry in log if entry.get("reward") is not None]
    if rewards:
        save_figure(
            make_reward_curve(list(range(len(rewards))), rewards),
            f"{out_dir}/grpo_reward_curve.png",
        )
        print("  wrote grpo_reward_curve.png")
except Exception as exc:
    print(f"  reward curve skipped: {exc}")

# 2) 6 reward components over training (sidecar from grpo_reward.py)
try:
    bds = get_recent_breakdowns()
    if bds:
        components = {
            k: [bd.to_dict()[k] for bd in bds]
            for k in ["diagnostic", "correct_op", "resolution",
                      "format", "efficiency", "penalty"]
        }
        # Smooth into moving average
        W = 32
        smoothed = {
            k: [sum(v[max(0, i-W):i+1])/min(i+1, W) for i in range(len(v))]
            for k, v in components.items()
        }
        save_figure(
            make_reward_components(list(range(len(bds))), smoothed),
            f"{out_dir}/grpo_reward_components.png",
        )
        print("  wrote grpo_reward_components.png")
except Exception as exc:
    print(f"  reward components skipped: {exc}")

# 3) Success bars: random vs SFT+GRPO across families
save_figure(
    make_success_bars(
        reports_by_condition={
            "random":   report_random_final.by_family,
            "sft+grpo": report_grpo.by_family,
        },
        families=EVAL_FAMILIES,
    ),
    f"{out_dir}/final_success_rates.png",
)
print("  wrote final_success_rates.png")

# 4) Action distributions
def _actions(report):
    out = defaultdict(int)
    for ep in report.episodes:
        for a, _ in ep.actions:
            out[a] += 1
    return dict(out)

save_figure(
    make_action_distribution(actions_by_condition={
        "random":   _actions(report_random_final),
        "sft+grpo": _actions(report_grpo),
    }),
    f"{out_dir}/final_action_distribution.png",
)
print("  wrote final_action_distribution.png")

# 5) Eval summary JSON for the README
summary = {
    "model": "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    "lora_r": 16,
    "lora_alpha": 32,
    "grpo_steps": GRPO_STEPS,
    "eval_seeds": FINAL_SEEDS,
    "by_family": {
        fam: {
            "random_success":   report_random_final.by_family[fam]["success_rate"],
            "sft+grpo_success": report_grpo.by_family[fam]["success_rate"],
            "random_avg_score":   report_random_final.by_family[fam]["avg_score"],
            "sft+grpo_avg_score": report_grpo.by_family[fam]["avg_score"],
        }
        for fam in EVAL_FAMILIES
    },
}
with open(f"{out_dir}/eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"  wrote eval_summary.json")

print(f"\nAll plots written to {out_dir}/ in {time.monotonic()-_t0:.1f}s")


## Cell 11 — Push LoRA adapter to your HuggingFace Hub (optional)

**Expected runtime: 2–3 minutes.** Skipped if the `HF_USER` and `HF_TOKEN`
environment variables aren't set. Set them in Colab via:
> Sidebar → 🔑 Secrets → add `HF_TOKEN` (write scope) and `HF_USER`.


In [ ]:
import os, time
_t0 = time.monotonic()

HF_USER = os.environ.get("HF_USER", "")
HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_USER or not HF_TOKEN:
    print("HF_USER / HF_TOKEN not set — skipping Hub push.")
    print("To enable: Colab sidebar → 🔑 Secrets → add both, then re-run this cell.")
else:
    from huggingface_hub import login, create_repo
    login(token=HF_TOKEN)
    repo_id = f"{HF_USER}/praetor-incident-commander-grpo"
    create_repo(repo_id, exist_ok=True)
    model.push_to_hub(repo_id, token=HF_TOKEN)
    tokenizer.push_to_hub(repo_id, token=HF_TOKEN)
    print(f"Pushed LoRA + tokenizer to https://huggingface.co/{repo_id}")
    print(f"  (paste this URL into the README's 'Trained LoRA adapter' line)")

print(f"\nDone in {time.monotonic()-_t0:.1f}s")


## Cell 12 — Print the README results table

**Expected runtime: < 1 second.** Copy this block straight into the
README's *Eval results* section, replacing the placeholder rows.


In [ ]:
print()
print("Paste the block below into README.md → 'Eval results' section:")
print()
print("=" * 70)
print()

print("| Condition | OOM Crash | DB Pool | Bad Deploy | Disk Full | Slow Query | Cert Expiry | Average |")
print("|---|---:|---:|---:|---:|---:|---:|---:|")

def _row(name, by_fam, bold=False):
    rates = [by_fam.get(f, {}).get("success_rate", 0) * 100 for f in EVAL_FAMILIES]
    avg = sum(rates) / len(rates)
    label = f"**{name}**" if bold else name
    cells = " | ".join(f"{r:>3.0f}%" for r in rates)
    print(f"| {label} | {cells} | **{avg:>3.0f}%** |")

_row("Random (n=60)", report_random_final.by_family)
_row("SFT + GRPO (n=60)", report_grpo.by_family, bold=True)

print()
print("=" * 70)
print()
print("Plot files (also commit these into results/ on GitHub):")
print("  /content/results/grpo_reward_curve.png")
print("  /content/results/grpo_reward_components.png")
print("  /content/results/final_success_rates.png")
print("  /content/results/final_action_distribution.png")
print("  /content/results/eval_summary.json")
